### HRSMC Course: Applied machine learning for chemistry
# Day 2, Introduction to Machine Learning
### by Bernd Ensing, University of Amsterdam

## Assignment 2 - Neural networks

In this tutorial, you will learn to make, train, and use neural network models. We will first implement a relatively small neural network, with one hidden layer, from scratch and apply it to the task of 1D regression. Next, to design and work with larger networks for the purpose of deep learning, you will learn to make use of the machine learning library [pytorch](https://pytorch.org). After getting acquainted with the essential pytorch modules, you will build a convolutional neural network (CNN) to classify images.


Tasks to fulfill:
* part 1
    * recal 1D regression using a polynomial fit function
    * build a 3-layer neural network from scratch
    * apply the network for non-linear regression
* part 2
    * learn to use the pytorch library in four steps
* part 3
    * build a CNN with pytorch
    * classify images


# Part 1: Implementing a neural network with one hidden layer

In part one, we will implement a neural network and train it to do one-dimensional regression. The neural network will consist of the input layer, one hidden layer, and the output layer.

To refresh our memory about 1D regression, let us first consider again optimizing a polynomial fit function using gradient descent.

To keeps things simple, we will aim to fit a simple sine function using 2000 points (i.e. we ignore noise).

Run the cells below to create the data arrays, $x$ and $y$, and plot the result. 

In [ ]:
# First load some useful libraries
import numpy as np
import matplotlib.pyplot as plt
import math
import torch

In [ ]:
# Create random input and output data
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

plt.figure()
plt.plot(x, y, 'g',linewidth=2.5, label='sin(x)')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

### 3rd order polynomial function

The first model that we will train on the above training data is a 3rd order polynomial function:

$$y = w_0 + w_1 x + w_2 x^2 + w_3 x^3$$

$\color{DarkBlue}{\textbf{Task:}}$ Complete in the next cell the function <code>my_model()</code>, which receives through its arguments an input array, $x$, and the four weights, and should return the output $y$-value.

In [ ]:
# compute the polynomial values
def my_model_pol(x: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
# ======== start your code here =================================
    y = torch.zeros_like(x)

    for i in range(len(weights)):
        y += weights[i] * torch.pow(x, i)
# ======== end your code here ===================================
    return y

Our model prediction for the $y$-value will be computed using the above function in the so-called _forward_ pass. To optimize the weights, we will need the gradients of the polynomial function during a _back propagation_ pass. 

$\color{DarkBlue}{\textbf{Task:}}$ Complete the code in the next cell by implementing the derivatives of the polynomial w.r.t. the weights.

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> it is important that all grad_w arrays will have the same length as $x$. For grad_w0, this is not automatic, as it does not directly depend on $x$. Use <code>np.ones(len(x))</code> to ensure that grad_w0 has the correct length.
    </li>    
</ul>
</details>

In [ ]:
# compute the derivatives of the polynomial w.r.t. the weights
def my_grad_pol(x: torch.Tensor, weights: torch.Tensor):
    # Initialize grads with shape (N_weights, x_datapoint)
    grads = torch.zeros((len(weights), len(x)), device=x.device, dtype=x.dtype)

    # Compute gradients for each weight and data point
    for i in range(len(weights)):
        grads[i] = torch.pow(x, i)  # Element-wise power for each data point

    return grads


Use the cell below to initialize the weights with "random" numbers and to check the <code>my_model()</code> and <code>my_grad</code> functions.

In [ ]:
# initialize weights randomly
weights = torch.randn(4, dtype=torch.float32)
# w0 = -1.3322530705186613
# w1 = -0.6327741012971428
# w2 = -0.7984372216326304
# w3 = 1.502203462916583

# test my_model() and my_grad()
xx = torch.tensor([1.4], dtype=torch.float32).reshape(-1, 1)
print(f" y = {my_model_pol(xx, weights)} | expected: 0.33897254 for the above, predefined weights")
print(f" y = {my_grad_pol(xx, weights)} | expected: 1., 1.4, 1.96, 2.744")

For the loss function, we will use the sum of squared differences between the target $y$-values and our predicted $y$-values.

$\color{DarkBlue}{\textbf{Task:}}$ 
* Implement the calculation of the loss in the next cell and run the optimization
* Plot the results in the following cell

In [ ]:

# set learning rate and number of cycles
learning_rate = 1e-6
ncycles = 2000

# main optimization loop
for t in range(ncycles):
    # Forward pass: compute predicted y
    y_pred = my_model_pol(x, weights)

    # Compute and print loss
# ======== start your code here =================================
    loss = torch.square(y_pred - y).sum()
# ======== end your code here ===================================
    if t % 100 == 99:
        print(t, loss.item())

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grads = my_grad_pol(x, weights)

    # apply chain rule and sum over the training batch
    for i in range(len(weights)):
        grad_wi = (grad_y_pred * grads[i]).sum()

        # Update weights
        weights[i] -= learning_rate * grad_wi

res_str = " + ".join([f"{weights[i].item():10.6f} x^{i}" for i in range(len(weights))])

print(f'Result: y = {res_str}')

In [ ]:
# plot ground truth function together with prediction with matplotlib

plt.figure(figsize=(6, 5))
plt.plot(x, y, 'g',linewidth=2.5, label='sin(x)')
plt.plot(x, y_pred, 'r',linewidth=2.5, label='prediction')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()


The above plot should show you a very reasonable fit of the sine function by the 3rd order polynomial. And the loss after 2000 cycles should be around 10. In the first tutorial, we already looked carefully at overfitting, regularization, and validation. We will skip that here, as the purpose of this exercise thus far is simply to remember the structure of the optimization procedure, consisting of: 
1. forward pass,
1. evaluate loss function,
1. back propagation,
1. update the weights.

We now continue to implement a neural network to do the job of 1D regression.

## Neural network

The neural network that we will implement has three layers: the input layer, which has one node for the input $x$, the hidden layer with $M$ nodes, and the output layer, which has one output node for $y$.

The general formula for such a fully connected feed forward network is:
$$ y(\mathbf{x},\mathbf{w}^{(1)},\mathbf{w}^{(2)})=\sum_{m=0}^M w_m^{(2)} h\big(\sum_{d=0}^D w_{md}^{(1)}\,x_d\big) $$

Here, $M$ is the number of nodes in the hidden layer. The number of input nodes, $D$, is 1, and we can made the bias, $\mathbf{w}^{(0)}$ explicit, so that the second sum disappears:

$$ y(\mathbf{x},\mathbf{w}^{(0)}, \mathbf{w}^{(1)}, \mathbf{w}^{(2)}) = \sum_{m=0}^M w_m^{(2)} h\big( w_{m}^{(1)} \, x + w_{m}^{(0)}\big) $$

For the activation function, $h$, we will use a sigmoid function, given by:
$$ h(x) = \frac{1}{1 + e^{-x}}$$


$\color{DarkBlue}{\textbf{Tasks:}}$ 
* First, complete in the next cell, the function to compute the sigmoid function value.
* In the next cell, the derivative of the sigmoid is computed. This looks surprisingly simple. Can you show (for yourself) that this is indeed the derivative of the sigmoid?
* Next, in the following cell, complete the modified <code>my_model()</code> function to compute the $y$-output of the neural network.


In [ ]:
# compute the sigmoid activation function value
def sigmoid(x):
# ======== start your code here =================================
    y = 1 / (1 + np.exp(-x))
# ======== end your code here ===================================
    return y


In [ ]:
# differential of the sigmoid
def grad_sigmoid(x):
    s = sigmoid(x)
    return s * (1 - s)

In [ ]:
# compute the neural network values
# strategy: compute first the value of the first node,
# then for-loop over the additional nodes and add their values
# Note: nhidden = M
def my_model_nn(x: np.ndarray | torch.Tensor, weights: np.ndarray | torch.Tensor, nhidden: int):
    y = weights[2][0] * sigmoid(weights[1][0] * x + weights[0][0])
    for i in range(1,nhidden):
        y += weights[2][i] * sigmoid(weights[1][i] * x + weights[0][i])
    return y


In [ ]:
# compute the derivatives of the net w.r.t. the weights
def my_grad_nn(x: np.ndarray | torch.Tensor, weights: np.ndarray | torch.Tensor):
    grad_w0 = weights[2] * grad_sigmoid(weights[1] * x + weights[0])
    grad_w1 = weights[2] * grad_sigmoid(weights[1] * x + weights[0]) * x
    grad_w2 = sigmoid(weights[1] * x + weights[0])
    return grad_w0, grad_w1, grad_w2


Run the next cell to test your model and gradients:

In [ ]:
# Randomly initialize weights
nhidden = 2
# w0 = np.random.randn(nhidden)
# w1 = np.random.randn(nhidden)
# w2 = np.random.randn(nhidden)
w0 = np.array([0.1265674, -1.47488376])
w1 = np.array([0.08898376, -1.3398663])
w2 = np.array([-0.99809171, -0.92373637])

weights = np.stack([w0, w1, w2])

xx = np.array([-1.4,1.4])
print(f" s = {sigmoid(xx)} | expected = [0.19781611 0.80218389]")
print(f" y = {my_model_nn(xx, weights, nhidden)} | expected = [-1.05277818 -0.59267402]")
grad_w0, grad_w1, grad_w2 = my_grad_nn(xx, weights)
print(f" dy/dw0 = {grad_w0} | expected = [-0.24952268 -0.24562934]")
print(f" dy/dw1 = {grad_w1} | expected = [ 0.34933175 -0.34388108]")
print(f" dy/dw2 = {grad_w2} | expected = [0.50049753 0.56245822]")

Finally, we run again the optimization routine containing the same four elements (forward pass, loss function, back propagation, and weights update) as before.

$\color{DarkBlue}{\textbf{Task:}}$ Complete the code in the next cell with the appropriate calls in the forward and backward passes.


In [ ]:
# Randomly initialize weights
nhidden = 4
weights = np.random.randn(3, nhidden)
grads = np.zeros((3, nhidden))

# set learning rate and number of cycles
learning_rate = 2e-5
ncycles = 20000

loss_values = []

# main optimization loop
for t in range(ncycles):
    # Forward pass: compute predicted y
# ======== start your code here =================================
    y_pred = my_model_nn(x, weights, nhidden)
# ======== end your code here ===================================

    # Compute and print loss
    loss = np.square(y_pred - y).sum()
    loss_values.append(loss)
    if t % 1000 == 999:
        print(f"Loss at iteration {t}: {loss :.4f}")

    if loss < 1.0:
        print(f"Converged with {nhidden} hidden nodes at iteration {t} with loss {loss :.4f}")
        break

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    for i in range(0, nhidden):
# ======== start your code here =================================
        grad_weights = my_grad_nn(x, weights[:, i])
        # apply chain rule and sum over the training batch
        grads[0, i] = (grad_y_pred * grad_weights[0]).sum()
        grads[1, i] = (grad_y_pred * grad_weights[1]).sum()
        grads[2, i] = (grad_y_pred * grad_weights[2]).sum()
# ======== end your code here ===================================

    # Update weights
    weights -= learning_rate * grads


In [ ]:
# plot the loss values over iterations
plt.figure(figsize=(6, 5))
plt.plot(loss_values, 'b',linewidth=2.5, label='loss')
plt.legend()
plt.xlabel('iteration',fontsize=20)
plt.ylabel('loss',fontsize=20)
plt.xticks(np.arange(0, len(loss_values), len(loss_values)//5), fontsize=14)
plt.yticks(fontsize=16)
plt.show()

In [ ]:
# plot ground truth function together with prediction with matplotlib

ys = np.zeros([nhidden, len(x)])
for i in range(nhidden):
    ys[i] = weights[2, i] * sigmoid(weights[1, i] * x + weights[0, i])

plt.figure(figsize=(6, 5))
plt.plot(x, y, 'g',linewidth=2.5, label='sin(x)')
plt.plot(x, y_pred, 'r',linewidth=2.5, label='prediction')
for i in range(nhidden):
    plt.plot(x, ys[i],linewidth=1.0, label=f'sigmoid{i}')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()

The number of nodes in the hidden layer and the number of optimization cycles are hyper-parameters.

$\color{DarkBlue}{\textbf{Question 1}}$
* How many hidden nodes and how many cycles do you need to get a loss smaller than 1.0?

$\color{Grey}{\textsf{1. Hidden Nodes: 4 | Iterations: 51483}}$ 
$\color{Grey}{\textsf{2. Hidden Nodes: 6 | Iterations: 25247}}$

# Part 2: Introduction to PyTorch

Thus far, we have mostly used NumPy to work with arrays and implement our first neural network. However, for larger networks, it is advantageous to make use of one of the available python packages made for working with neural networks. In this part, we will get acquainted with the machine learning package [PyTorch](https://pytorch.org), in four steps entitled:
1. Tensors
1. Autograd
1. NN module
1. Optim module

To introduce each of these concepts, we will use as an example again the 1D regression task, using the 3rd order polynomial fit function. At the end of this part, we will have converted the NumPy implementation to a code that makes full use of the PyTorch features.

For further information on PyTorch, here are a few helpful links:
* [Introduction to PyTorch](https://pytorch.org/tutorials/beginner/introyt/introyt1_tutorial.html)
* [Youtube series on PyTorch](https://pytorch.org/tutorials/beginner/introyt.html)
* [PyTorch cheat sheet](https://pytorch.org/tutorials/beginner/ptcheat.html)
* [PyTorch documentation](https://pytorch.org/docs/stable/index.html)


## PyTorch 1: Tensors

Numpy is a great framework, but it cannot utilize GPUs to accelerate its numerical computations. For modern deep neural networks, GPUs often provide speedups of 50x or greater, so unfortunately numpy won’t be enough for modern deep learning.

Here we introduce the most fundamental PyTorch concept: the Tensor. A PyTorch Tensor is conceptually identical to a numpy array: a Tensor is an n-dimensional array, and PyTorch provides many functions for operating on these Tensors. Behind the scenes, Tensors can keep track of a computational graph and gradients, but they’re also useful as a generic tool for scientific computing.

Also unlike numpy, PyTorch Tensors can utilize GPUs to accelerate their numeric computations. To run a PyTorch Tensor on GPU, you simply need to specify the correct device.

Here we use PyTorch Tensors to fit a third order polynomial to sine function. Like the numpy example above we need to manually implement the forward and backward passes through the network:

$\color{DarkBlue}{\textbf{Task:}}$
* Complete the code in the next cell, by adding the necessary declarations, calculations, and updates for weights w1, w2, and w3 (note: for w0, this has already been done).
* Run the code in the next cell
* Find in the PyTorch documentation how to set the random number seed and adapt the code (where indicated) to make the output reproducible




In [ ]:
import math

# set torch random number seed for reproducibility
torch.manual_seed(0)

dtype = torch.float
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Create random input and output data
x = torch.linspace(-math.pi, math.pi, 2000, device=device, dtype=dtype)
y = torch.sin(x)

# Randomly initialize weights
weights = torch.randn(4, device=device, dtype=dtype)
# ======== end your code here ===================================

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    y_pred = my_model_pol(x, weights)

    # Compute and print loss
    loss = (y_pred - y).pow(2).sum().item()
    if t % 100 == 99:
        print(f"t = {t}, loss = {loss :.4f}")

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_weights = my_grad_pol(x, weights)

    # ======== start your code here =================================
    for i in range(len(weights)):
        # apply chain rule and sum over the training batch
        grad_wi = (grad_y_pred * grad_weights[i]).sum()

        # Update weights using gradient descent
        weights[i] -= learning_rate * grad_wi
    # ======== end your code here ===================================

res_str = " + ".join([f"{weights[i].item():10.6f} x^{i}" for i in range(len(weights))])

print(f'Result: y = {res_str}')

In [ ]:
# plot ground truth function together with prediction with matplotlib
plt.figure(figsize=(6, 5))
plt.plot(x.cpu(), y.cpu(), 'g',linewidth=2.5, label='sin(x)')
plt.plot(x.cpu(), y_pred.detach().cpu(), 'r',linewidth=2.5, label='prediction')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()


## PyTorch 2: autograd

In the above examples, we had to manually implement both the forward and backward passes of our neural network. Manually implementing the backward pass is not a big deal for a small two-layer network, but can quickly get very hairy for large complex networks.

Thankfully, we can use automatic differentiation to automate the computation of backward passes in neural networks. The autograd package in PyTorch provides exactly this functionality. When using autograd, the forward pass of your network will define a computational graph; nodes in the graph will be Tensors, and edges will be functions that produce output Tensors from input Tensors. Backpropagating through this graph then allows you to easily compute gradients.

This sounds complicated, it’s pretty simple to use in practice. Each Tensor represents a node in a computational graph. If x is a Tensor that has x.requires_grad=True then x.grad is another Tensor holding the gradient of x with respect to some scalar value.

Here we use PyTorch Tensors and autograd to implement our fitting sine wave with third order polynomial example; now we no longer need to manually implement the backward pass through the network.

$\color{DarkBlue}{\textbf{Task:}}$
* Find the differences between the following code and the previous one.

In [ ]:
import torch
import math

# set torch random number seed for reproducibility
torch.manual_seed(0)

# Create Tensors to hold input and outputs.
# By default, requires_grad=False, which indicates that we do not need to
# compute gradients with respect to these Tensors during the backward pass.
x = torch.linspace(-math.pi, math.pi, 2000, device=device, dtype=dtype)
y = torch.sin(x)

# Create random Tensors for weights. For a third order polynomial, we need
# 4 weights: y = a + b x + c x^2 + d x^3
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors during the backward pass.
weights = torch.randn(4, device=device, dtype=dtype, requires_grad=True)

learning_rate: float = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y using operations on Tensors.
    y_pred = my_model_pol(x, weights)

    # Compute and print loss using operations on Tensors.
    # Now loss is a Tensor of shape (1,)
    # loss.item() gets the scalar value held in the loss.
    loss: torch.Tensor = (y_pred - y).pow(2).sum()
    if t % 100 == 99:
        print(t, loss.item())

    # Use autograd to compute the backward pass. This call will compute the
    # gradient of loss with respect to all Tensors with requires_grad=True.
    # After this call a.grad, b.grad. c.grad and d.grad will be Tensors holding
    # the gradient of the loss with respect to a, b, c, d respectively.
    loss.backward()

    # Manually update weights using gradient descent. Wrap in torch.no_grad()
    # because weights have requires_grad=True, but we don't need to track this
    # in autograd.
    with torch.no_grad():
        weights -= learning_rate * weights.grad

        # Manually zero the gradients after updating weights
        weights.grad = None

res_str = " + ".join([f"{weights[i].item():10.6f} x^{i}" for i in range(len(weights))])
print(f"Result: y = {res_str}")

In [ ]:
# plot ground truth function together with prediction with matplotlib
plt.figure(figsize=(6, 5))
plt.plot(x, y, 'g',linewidth=2.5, label='sin(x)')
plt.plot(x, y_pred.detach(), 'r',linewidth=2.5, label='prediction')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()

## PyTorch 3: the nn module

Computational graphs and autograd are a very powerful paradigm for defining complex operators and automatically taking derivatives; however for large neural networks raw autograd can be a bit too low-level.

When building neural networks we frequently think of arranging the computation into layers, some of which have learnable parameters which will be optimized during learning.

In TensorFlow, packages like Keras, TensorFlow-Slim, and TFLearn provide higher-level abstractions over raw computational graphs that are useful for building neural networks.

In PyTorch, the nn package serves this same purpose. The nn package defines a set of Modules, which are roughly equivalent to neural network layers. A Module receives input Tensors and computes output Tensors, but may also hold internal state such as Tensors containing learnable parameters. The nn package also defines a set of useful loss functions that are commonly used when training neural networks.

In this example we use the nn package to implement our polynomial model network.

$\color{DarkBlue}{\textbf{Task:}}$
* Modify the code below to use a 4th order polynomial as your model instead of the current 3rd order polynomial.

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> Remember that you may have to adapt the learning rate when you increase the complexity of the model.
    </li>    
</ul>
</details>

In [ ]:
import torch
import math

# set torch random number seed for reproducibility
torch.manual_seed(0)

# Create Tensors to hold input and outputs.
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

# For this example, the output y is a linear function of (x, x^2, x^3), so
# we can consider it as a linear layer neural network. Let's prepare the
# tensor (x, x^2, x^3).
# ======== modify the code here =================================
p = torch.tensor([1, 2, 3])
# ======== end your code here ====================================
xx = x.unsqueeze(-1).pow(p)


# In the above code, x.unsqueeze(-1) has shape (2000, 1), and p has shape
# (3,), for this case, broadcasting semantics will apply to obtain a tensor
# of shape (2000, 3)

# Use the nn package to define our model as a sequence of layers. nn.Sequential
# is a Module which contains other Modules, and applies them in sequence to
# produce its output. The Linear Module computes output from input using a
# linear function, and holds internal Tensors for its weight and bias.
# The Flatten layer flatens the output of the linear layer to a 1D tensor,
# to match the shape of `y`.

# ======== modify the code here =================================
model = torch.nn.Sequential(
    torch.nn.Linear(3, 1),
    torch.nn.Flatten(0, 1)
)
# ======== end your code here ===================================



# The nn package also contains definitions of popular loss functions; in this
# case we will use Mean Squared Error (MSE) as our loss function.
loss_fn = torch.nn.MSELoss(reduction='sum')

learning_rate = 1e-6
for t in range(2000):

    # Forward pass: compute predicted y by passing x to the model. Module objects
    # override the __call__ operator so you can call them like functions. When
    # doing so you pass a Tensor of input data to the Module and it produces
    # a Tensor of output data.
    y_pred = model(xx)

    # Compute and print loss. We pass Tensors containing the predicted and true
    # values of y, and the loss function returns a Tensor containing the
    # loss.
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())

    # Zero the gradients before running the backward pass.
    model.zero_grad()

    # Backward pass: compute gradient of the loss with respect to all the learnable
    # parameters of the model. Internally, the parameters of each Module are stored
    # in Tensors with requires_grad=True, so this call will compute gradients for
    # all learnable parameters in the model.
    loss.backward()

    # Update the weights using gradient descent. Each parameter is a Tensor, so
    # we can access its gradients like we did before.
    with torch.no_grad():
        for param in model.parameters():
            param -= learning_rate * param.grad

# You can access the first layer of `model` like accessing the first item of a list
linear_layer = model[0]

# For linear layer, its parameters are stored as `weight` and `bias`.
print(f'Result: y = {linear_layer.bias.item():10.6f} + {linear_layer.weight[:, 0].item():10.6f} x + {linear_layer.weight[:, 1].item():10.6f} x^2 + {linear_layer.weight[:, 2].item():10.6f} x^3')



In [ ]:
# plot ground truth function together with prediction with matplotlib
plt.figure(figsize=(6, 5))
plt.plot(x, y, 'g',linewidth=2.5, label='sin(x)')
plt.plot(x, y_pred.detach(), 'r',linewidth=2.5, label='prediction')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()

## PyTorch 4: the optim module

Up to this point we have updated the weights of our models by manually mutating the Tensors holding learnable parameters with torch.no_grad(). This is not a huge burden for simple optimization algorithms like stochastic gradient descent, but in practice we often train neural networks using more sophisticated optimizers like AdaGrad, RMSProp, Adam, etc.

The optim package in PyTorch abstracts the idea of an optimization algorithm and provides implementations of commonly used optimization algorithms.

In this example we will use the nn package to define our model as before, but we will optimize the model using the RMSprop algorithm provided by the optim package.

$\color{DarkBlue}{\textbf{Task:}}$
* Modify the code to use instead of the <code>RMSprop()</code> optimizer, the <code>LBFGS</code> optimizer, by commenting out the second and commenting the first.



In [ ]:
import torch
import math


# Create Tensors to hold input and outputs.
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

# Prepare the input tensor (x, x^2, x^3).
p = torch.tensor([1, 2, 3])
xx = x.unsqueeze(-1).pow(p)

# Use the nn package to define our model and loss function.
model = torch.nn.Sequential(
    torch.nn.Linear(3, 1),
    torch.nn.Flatten(0, 1)
)
loss_fn = torch.nn.MSELoss(reduction='sum')

# Use the optim package to define an Optimizer that will update the weights of
# the model for us. Here we will use RMSprop; the optim package contains many other
# optimization algorithms. The first argument to the RMSprop constructor tells the
# optimizer which Tensors it should update.
learning_rate = 1e-3
# optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)
optimizer = torch.optim.LBFGS(model.parameters(), lr=learning_rate)

for t in range(2000):
    # Forward pass: compute predicted y by passing x to the model.
    y_pred = model(xx)

    # Compute and print loss.
    loss = loss_fn(y_pred, y)
    if t % 100 == 99:
        print(t, loss.item())

    # Before the backward pass, use the optimizer object to zero all of the
    # gradients for the variables it will update (which are the learnable
    # weights of the model). This is because by default, gradients are
    # accumulated in buffers( i.e, not overwritten) whenever .backward()
    # is called. Checkout docs of torch.autograd.backward for more details.
    optimizer.zero_grad()

    # Backward pass: compute gradient of the loss with respect to model
    # parameters
    loss.backward()

    # Calling the step function on an Optimizer makes an update to its
    # parameters
    # optimizer.step()
    optimizer.step(lambda: loss_fn(y_pred, y))

linear_layer = model[0]
print(f'Result: y = {linear_layer.bias.item():10.6f} + {linear_layer.weight[:, 0].item():10.6f} x + {linear_layer.weight[:, 1].item():10.6f} x^2 + {linear_layer.weight[:, 2].item():10.6f} x^3')



In [ ]:
# plot ground truth function together with prediction with matplotlib
plt.figure(figsize=(6, 5))
plt.plot(x, y, 'g',linewidth=2.5, label='sin(x)')
plt.plot(x, y_pred.detach(), 'r',linewidth=2.5, label='prediction')
plt.legend()
plt.xlabel('x',fontsize=20)
plt.ylabel('y',fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.show()

## Conclusions part 1

You reached the end of this part of the Tutorial on Neural Networks! Well done!

You have implemented your first neural network from scratch and applied it to the task of regression. Secondly, you learned the rudiments of the PyTorch package.

In the second part of this tutorial, we will make use of PyTorch to make a deep learning model for classification.

In case you would like to first review some more the essentials of PyTorch, here are once more the links to the documentation and the youtube videos:

* [Introduction to PyTorch](https://pytorch.org/tutorials/beginner/introyt/introyt1_tutorial.html)
* [Youtube series on PyTorch](https://pytorch.org/tutorials/beginner/introyt.html)
* [PyTorch cheat sheet](https://pytorch.org/tutorials/beginner/ptcheat.html)
* [PyTorch documentation](https://pytorch.org/docs/stable/index.html)


 


## Neural networks - part 2

In the second part of this tutorial, you learn to make, train, and use neural network models. Having implemented a relatively small neural network from scratch and applied it to the task of 1D regression and getting acquainted with the essential pytorch modules, here we will 
build with pytorch a convolutional neural network (CNN) to classify images.


Tasks to fulfill:
* part 3
    * build a CNN with pytorch
    * classify images

In [ ]:
# first we load some useful libraries
import torch
from torchvision import transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np

## Fetching the MNIST dataset and preparing train, validation and test data loaders

The images that we will classify are from the famous MNIST database of handwritten digits, [available from this page](http://yann.lecun.com/exdb/mnist/). It has a training set of 60,000 examples, and a test set of 10,000 examples. It is a subset of a larger set available from NIST. The digits have been size-normalized and centered in a fixed-size image.

We can download the dataset from the NIST website but it is more convenient to use the dataset API under torchvision that PyTorch provides, which directly fetches the MNIST data.

We fetch both training and test set made available by NIST. For fetching the training set we set the <code>train</code> argument to <code>True</code> whereas to fetch the test set we set it to <code>False</code>. The <code>dataset</code> API also allows us to address any transformations we want to apply to the data. We set <code>transform=transforms.Compose([transforms.ToTensor()]))</code> to convert the image data to tensors.

$\color{DarkBlue}{\textbf{Task:}}$ Execute the next cell to download the MNIST data.

In [ ]:
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.Compose([transforms.ToTensor()]))
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.Compose([transforms.ToTensor()]))

# split the test dataset into two sets one for validation and
# the other for testing. Division: 90%–10% i.e. 9000 for validation and 1000 for testing
mnist_valset, mnist_testset = torch.utils.data.random_split(mnist_testset, [int(0.9 * len(mnist_testset)), int(0.1 * len(mnist_testset))])

# prepare data loaders for all three sets.
train_dataloader = torch.utils.data.DataLoader(mnist_trainset, batch_size=64, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(mnist_valset, batch_size=32, shuffle=False)
test_dataloader = torch.utils.data.DataLoader(mnist_testset, batch_size=32, shuffle=False)

$\color{DarkBlue}{\textbf{Question 1:}}$ The variable <code>mnist_trainset</code> is a 2-dimensional array. What are the dimensions of <code>mnist_trainset</code>? What could be the contents of <code>mnist_trainset[0][0]</code> and of <code>mnist_trainset[0][1]</code>?

$\color{Grey}{\textsf{<double-click and type your answer here>}}$

The three dataLoaders defined at the end basically combine a dataset and a sampler, and provides an iterable over the given dataset.

It also allows us to pick batch sizes. The batch size is a hyperparameter that defines the number of samples the model looks at before updating the internal model parameters. This concept is called mini-batch gradient descent as the model works on small batches of data to calculate gradients and modifies the model parameters based off them. We choose a batch size of 64 for the train_dataloader, batch size of 32 for val_dataloader and test_dataloader.

## Visualizing the data
One of the first step while developing a deep learning model is to visualize the data. So let’s plot a few images from the trainset.

In [ ]:
# visualize data
fig=plt.figure(figsize=(20, 10))
for i in range(1, 6):
    img = transforms.ToPILImage(mode='L')(mnist_valset[i][0])
    fig.add_subplot(1, 6, i)
    plt.title(mnist_valset[i][1])
    plt.imshow(img)
plt.show()

This plot shows the digit image along with it’s label on the top. If you print the image data out you can see the values are between 0 and 1.

## Defining the Model
We will define a simple convolutional neural network with 2 convolution layers followed by two fully connected layers.

Below is the model architecture we will be using for our CNN.

<img src="cnn.png">

We follow up each convolution layer with a RelU activation function and a max-pool layer. RelU introduces non-linearity and max-pooling helps with removing noise.

The first convolution layer <code>self.conv_1</code> takes in a channel of dimension 1 since the images are grayscaled. The kernel size is chosen to be of size 3x3 with stride of 1. The output of this convolution is set to 32 channels which means it will extract 32 feature maps using 32 kernels. We pad the image with a padding size of 1 so that the input and output dimensions are same. The output dimension at this layer will be 32 x 28 x 28.

We apply RelU activation to it followed by a max-pooling layer with kernel size of 2 and stride 2. This down-samples the feature maps to dimension of 32 x 14 x 14.

The second convolution layer <code>self.conv_2</code> will have an input channel size of 32. We choose an output channel size to be 64 which means it will extract 64 feature maps. The kernel size for this layer is 3 with stride 1. We again use a padding size of 1 so that the input and output dimension remain the same. The output dimension at this layer will be 64 x 7 x 7.

We then follow up it with a RelU activation and a max-pooling layer with kernel of size 2 and stride 2. The down-sampled feature map will have dimension 64 x 7 x 7.

We use the same definition <code>self.relu</code> and <code>self.max_pool2d</code> for this as for <code>self.relu</code> is the same operation and defining it twice makes no sense. Similarly <code>self.max_pool2d</code> is also the same operation as they use the same kernel size and stride count.

Finally, two fully connected layers <code>self.linear_1</code> and <code>self.linear_2</code> are used. We will pass a flattened version of the feature maps to the first fully connected layer. Hence it has to be of dimension 64 x 7 x 7 which is equal to 3136 nodes. The first linear layer is followed by a ReLU activation lauer. This layer will connect to another fully connected linear layer with 128 nodes. This will be our final layer so the output dimension should match the total classes which is 10. So we have two fully connected layers of size 3136 x 128 followed up by 128 x 10.

In between the two linear layers, we also want to dropout connections to reduce overfitting, which we define with <code>self.dropout</code> with a connection dropout probability of 0.5.

All the above layers and operations are defined under the <code>__init__</code> method of the class <code>torch.nn.Module</code>.

The <code>forward</code> method of the class <code>torch.nn.Module</code> defines how we want the data to flow through these layers. 

$\color{DarkBlue}{\textbf{Task:}}$
* Complete the network architecture definition in the <code>forward</code> function.

<details>
<summary> <font color='green'>Click here for hints</font></summary>
<ul>
    <li> To flatten the feature maps before the fully connected layers, you can use <code>x = x.reshape(x.size(0), -1)</code>.
    </li>    
</ul>
</details>

In [ ]:
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.conv_1 = torch.nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv_2 = torch.nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.max_pool2d = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        self.linear_1 = torch.nn.Linear(7 * 7 * 64, 128)
        self.linear_2 = torch.nn.Linear(128, 10)
        self.dropout = torch.nn.Dropout(p=0.5)
        self.relu = torch.nn.ReLU()

    def forward(self, x):
        x = self.conv_1(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
# ======== start your code here =================================


# ======== end your code here ===================================
        pred = x

        return pred

## Defining model object, loss function and optimizer
We define an instance of the <code>Model()</code> class and call it model.

__Loss Criterion__: Loss function tells us how good the model is predicting. Since we are dealing with multi class classification we choose cross entropy as our loss function. We use the <code>torch.nn.CrossEntropyLoss</code> which combines a softmax function <code>nn.LogSoftmax()</code> along with <code>nn.NLLLoss()</code> loss function.

__Optimizer__: An optimizer ties the loss function and model parameters together by updating the model in response to the output of the loss criterion. We will be using Adam as our optimizer with a learning rate of 0.001.


In [ ]:
model = Model()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

if (torch.cuda.is_available()):
    model.cuda()

## Training and Validation

We will train the model for a number <code>no_epochs</code> epochs and pick the model with the lowest validation lost. Since training is slow on a CPU, let's start with <code>no_epochs = 2</code>, although to arrive at a decent prediction, the number should be in between 20 and 100.

In [ ]:
no_epochs = 2
train_loss = list()
val_loss = list()
best_val_loss = 1

### Training loop
For each epoch we iterate through batches of train_dataset using an enumerate function over the train_dataloader.

First we set the model to train mode. This allows us to enable the dropout layer and also set the model in training mode with <code>model.train()</code>.

For each batch we pass the batch of image tensors into the model which will return a tensor having predictions for that batch (<code>pred = model(image)</code>).

Having got the predictions we pass them into the cross entropy loss criterion along with their actual labels and calculate the loss (<code>loss = criterion(pred, label)</code>).

We do a backward pass using this loss value and use Adam optimizer to modify the model parameters:
<code>loss.backward(), optimizer.step()</code>


Before each iteration we need to zero out the optimizer gradients (<code>optimizer.zero_grad()</code>).

We get the training loss for the entire epoch by adding all the losses for each batch iteration and averaging it over by the iteration count. Along with that we also keep track of the training loss at each epoch by storing the loss value in a list train_loss

In the code below, the training loop is already implemented.

### Validation loop

The second half of the main loop is a validation loop, which is done for each epoch after the training loop to see how that model performs on the validation set.

First the model is set to eval mode (<code>model.eval()</code>).

We then iterate over each batch from the validation dataloader using the enumerate function. We do similar steps as training but we do not backpropagate the loss. After that we compare the model’s prediction with the actual labels and calculate the accuracy of the model. Similar to the training phase, we get the validation loss for the epoch by adding all the losses for each batch iteration and averaging it over by the iteration count. We keep track of validation loss at each epoch by storing the loss value at each epoch in val_loss


In [ ]:
for epoch in range(no_epochs):

    total_train_loss = 0
    model.train()
    # training loop
    for itr, (image, label) in enumerate(train_dataloader):

        if (torch.cuda.is_available()):
            image = image.cuda()
            label = label.cuda()

        optimizer.zero_grad()

        pred = model(image)

        loss = criterion(pred, label)
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()

    total_train_loss = total_train_loss / (itr + 1)
    train_loss.append(total_train_loss)


    total_val_loss = 0
    total = 0
    model.eval()
    # validation loop
    for itr, (image, label) in enumerate(val_dataloader):

        if (torch.cuda.is_available()):
            image = image.cuda()
            label = label.cuda()

        pred = model(image)

        loss = criterion(pred, label)
        total_val_loss += loss.item()

        pred = torch.nn.functional.softmax(pred, dim=1)
        for i, p in enumerate(pred):
            if label[i] == torch.max(p.data, 0)[1]:
                total = total + 1

    accuracy = total / len(mnist_valset)

    total_val_loss = total_val_loss / (itr + 1)
    val_loss.append(total_val_loss)

    print('\nEpoch: {}/{}, Train Loss: {:.8f}, Val Loss: {:.8f}, Val Accuracy: {:.8f}'.format(epoch + 1, no_epochs, total_train_loss, total_val_loss, accuracy))

    if total_val_loss < best_val_loss:
        best_val_loss = total_val_loss
        print("Saving the model state dictionary for Epoch: {} with Validation loss: {:.8f}".format(epoch + 1, total_val_loss))
        torch.save(model.state_dict(), "model.dth")

We plot the training and validation loss for each epoch which are stored under liststrain_loss and val_loss respectively.

In [ ]:
fig=plt.figure(figsize=(20, 10))
plt.plot(np.arange(1, no_epochs+1), train_loss, label="Train loss")
plt.plot(np.arange(1, no_epochs+1), val_loss, label="Validation loss")
plt.xlabel('Loss')
plt.ylabel('Epochs')
plt.title("Loss Plots")
plt.legend(loc='upper right')
plt.show()

If you run the optimization long enough, you should see that at epoch no. 12 it has the lowest validation loss of 0.02092566 and accuracy of 0.99388889 after which the model starts overfitting and the validation loss explodes.

## Testing and Visualizing results

We fetch the saved model state dictionary for the model having the lowest validation loss and load it up into our model. Next we set the model to eval mode.

In [ ]:
# test model
model.load_state_dict(torch.load("model.dth"))
model.eval()

We enumerate through our test dataloader and calculate the models accuracy the same way we did with the validation loop. We also store the results in a “results” list.

In [ ]:
results = list()
total = 0
for itr, (image, label) in enumerate(test_dataloader):

    if (torch.cuda.is_available()):
        image = image.cuda()
        label = label.cuda()

    pred = model(image)
    pred = torch.nn.functional.softmax(pred, dim=1)

    for i, p in enumerate(pred):
        pred_label = torch.max(p.data, 0)[1]
        if label[i] == pred_label:
            total += 1
            # store just this image, not the whole batch
            results.append((image[i].detach().cpu(), pred_label.detach().cpu()))


We then plot the results.

In [ ]:
results[i][0].shape
print(total,i,itr)
print(pred.shape)

In [ ]:
test_accuracy = total / (itr + 1)
print('Test accuracy {:.8f}'.format(test_accuracy))

# visualize results
fig=plt.figure(figsize=(20, 10))
for i in range(1, 11):
    image = results[i][0].squeeze(0).detach().cpu()
    img = transforms.ToPILImage(mode='L')(image)
    fig.add_subplot(2, 5, i)
    plt.title(results[i][1].item())
    plt.imshow(img)
plt.show()


## Conclusions

This ends this part of the ML4Chem Tutorial on Neural Networks.

You have completed now a fully-functional convolutional neural network to classify images! Note that many aspects from the previous tutorials came together in this exercise, including training and validation, forward and backward propagation, and cross-entropy loss functions.

In practice, the performance can be further improved using data augmentation, tuning dropout layers, regularization, batch normalization, hyper-parameter tuning. etc.

